In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con).to_string(index=False))

In [ ]:
q1 = "SELECT COUNT(*) AS rows_total, SUM(analysed) AS analysed FROM matches"
print(pd.read_sql(q1, con).to_string(index=False))

q2 = """
SELECT t.name, COUNT(*) AS matches
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1 AND m.file_year >= 2015
GROUP BY t.name
ORDER BY matches DESC
"""
print(pd.read_sql(q2, con).to_string(index=False))

In [ ]:
#matches per year and per surface
q3 = """
SELECT file_year, surface, COUNT(*) AS matches
FROM matches
WHERE file_year >= 2015
GROUP BY file_year, surface
ORDER BY file_year, surface
"""
print(pd.read_sql(q3, con).to_string(index=False))

In [ ]:
# same as above but in a table 
df3 = pd.read_sql(q3, con)
print(df3.pivot(index="file_year", columns="surface", values="matches").to_string())

In [ ]:
#percentage of service points that ended in ace
q4 = """
SELECT t.name,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name
ORDER BY aces_pct DESC
"""
print(pd.read_sql(q4, con).to_string(index=False))

In [ ]:
#same above but evolution by years
q5 = """
SELECT t.name,
       m.file_year,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name, m.file_year
HAVING COUNT(*) >= 20
ORDER BY t.name, m.file_year
"""
df5 = pd.read_sql(q5, con)

print(df5.pivot(index="name", columns="file_year", values="aces_pct").to_string())
print()
print(df5.pivot(index="name", columns="file_year", values="matches_used").to_string())

In [ ]:
con.execute("DROP VIEW IF EXISTS serve_obs")
con.execute("""
CREATE VIEW serve_obs AS
SELECT match_id, file_year, event_no, analysed, surface, best_of, indoor,
       winner_id AS server_id, loser_id AS returner_id,
       w_svpt AS svpt, w_1stWon + w_2ndWon AS sv_won, w_ace AS ace
FROM matches
WHERE retired = 0
  AND w_svpt IS NOT NULL AND w_1stWon IS NOT NULL AND w_2ndWon IS NOT NULL
  AND w_ace IS NOT NULL AND w_svpt > 0
UNION ALL
SELECT match_id, file_year, event_no, analysed, surface, best_of, indoor,
       loser_id, winner_id,
       l_svpt, l_1stWon + l_2ndWon, l_ace
FROM matches
WHERE retired = 0
  AND l_svpt IS NOT NULL AND l_1stWon IS NOT NULL AND l_2ndWon IS NOT NULL
  AND l_ace IS NOT NULL AND l_svpt > 0
""")
con.commit()

In [ ]:
print(pd.read_sql("""
SELECT COUNT(*) AS rows_total,
       SUM(analysed) AS analysed_rows,
       SUM(CASE WHEN sv_won > svpt OR ace > svpt OR ace > sv_won THEN 1 ELSE 0 END) AS impossible_rows
FROM serve_obs
""", con).to_string(index=False))

print(pd.read_sql("""
SELECT surface,
       COUNT(*) AS rows_n,
       ROUND(100.0 * SUM(sv_won) / SUM(svpt), 2) AS service_pts_won_pct,
       ROUND(100.0 * SUM(ace) / SUM(svpt), 2) AS aces_pct
FROM serve_obs
WHERE file_year >= 2015
GROUP BY surface
ORDER BY service_pts_won_pct DESC
""", con).to_string(index=False))

In [ ]:
import numpy as np

obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)
tournament_names = pd.read_sql("SELECT event_no, name FROM tournaments", con)
print(obs.shape)


def window_sums(df, keys, prefix):
    base = df.groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    parts = [base.assign(file_year=base["file_year"] + d) for d in (-1, 0, 1)]
    out = pd.concat(parts).groupby(keys + ["file_year"], as_index=False)[["svpt", "val"]].sum()
    return out.rename(columns={"svpt": prefix + "_svpt", "val": prefix + "_val"})


def role_excess(t, d, pid_col, sf, prefix, mu, k):
    dd = d.assign(pid=d[pid_col])
    tt = t.assign(pid=t[pid_col])
    w = window_sums(dd, sf + ["pid"], prefix + "w")
    e = window_sums(dd, sf + ["pid", "event_no"], prefix + "e")
    tt = tt.merge(w, on=sf + ["pid", "file_year"], how="left")
    tt = tt.merge(e, on=sf + ["pid", "event_no", "file_year"], how="left")
    n = (tt[prefix + "w_svpt"].fillna(0) - tt[prefix + "e_svpt"].fillna(0)).to_numpy()
    v = (tt[prefix + "w_val"].fillna(0) - tt[prefix + "e_val"].fillna(0)).to_numpy()
    return (v - mu * n) / (n + k), n


def expected_values(obs, value, by_surface=True, k=500, analysed_only=True):
    sf = ["surface"] if by_surface else []
    d = obs.rename(columns={value: "val"})[["match_id", "file_year", "event_no", "analysed", "surface",
                                              "server_id", "returner_id", "svpt", "val"]]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)

    t = t.merge(window_sums(d, sf, "wt"), on=sf + ["file_year"], how="left")
    t = t.merge(window_sums(d, sf + ["event_no"], "et"), on=sf + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()

    ex_s, n_s = role_excess(t, d, "server_id", sf, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", sf, "r", mu, k)

    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t

In [ ]:
res_spw = expected_values(obs, "sv_won")
obs_pct = 100 * res_spw["val"].sum() / res_spw["svpt"].sum()
exp_pct = 100 * (res_spw["exp"] * res_spw["svpt"]).sum() / res_spw["svpt"].sum()
print(len(res_spw), "rows | observed", round(float(obs_pct), 2), "% | expected", round(float(exp_pct), 2), "%")
print("rows with no server baseline:", int((res_spw["n_srv"] == 0).sum()),
      "| no returner baseline:", int((res_spw["n_ret"] == 0).sum()))

In [ ]:
def speed_table(res, B=1000, min_matches=20, seed=1):
    m = res.groupby(["match_id", "event_no"], as_index=False).agg(
        rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
    m = m[m["rows"] == 2]
    rng = np.random.default_rng(seed)
    out = []
    for ev, g in m.groupby("event_no"):
        if len(g) < min_matches:
            continue
        s, r = g["svpt"].to_numpy(), g["resid"].to_numpy()
        idx = rng.integers(0, len(g), size=(B, len(g)))
        boots = 100 * r[idx].sum(axis=1) / s[idx].sum(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"event_no": ev, "matches": len(g),
                    "speed_pts": round(100 * r.sum() / s.sum(), 2),
                    "ci_low": round(float(lo), 2), "ci_high": round(float(hi), 2)})
    return (pd.DataFrame(out).merge(tournament_names, on="event_no")
            .sort_values("speed_pts", ascending=False)[["name", "matches", "speed_pts", "ci_low", "ci_high"]])

print("Service points won, relative to other events on the same surface")
print(speed_table(res_spw).to_string(index=False))

In [ ]:
res_ace = expected_values(obs, "ace")
print("Aces, relative to other events on the same surface")
print(speed_table(res_ace).to_string(index=False))
print()
res_spw_all = expected_values(obs, "sv_won", by_surface=False)
print("Service points won, relative to ALL other events (surfaces mixed)")
print(speed_table(res_spw_all).to_string(index=False))

In [ ]:
res_all = expected_values(obs, "sv_won", analysed_only=False)
o = 100 * res_all["val"].sum() / res_all["svpt"].sum()
e = 100 * (res_all["exp"] * res_all["svpt"]).sum() / res_all["svpt"].sum()
print(len(res_all), "rows | observed", round(float(o), 2), "% | expected", round(float(e), 2), "%")

In [ ]:
t3 = speed_table(res_spw_all)
print(t3.iloc[:7].to_string(index=False))
print(t3.iloc[7:].to_string(index=False))

In [ ]:
obs = pd.read_sql("""
    SELECT match_id, file_year, event_no, analysed, surface, indoor,
           server_id, returner_id, svpt, sv_won, ace
    FROM serve_obs
""", con)
print(obs.shape, obs["indoor"].value_counts(dropna=False).to_dict())

In [ ]:
def expected_values_k(obs, value, keys, k=500, analysed_only=True):
    base_cols = ["match_id", "file_year", "event_no", "analysed", "server_id", "returner_id", "svpt"]
    d = obs.rename(columns={value: "val"})[list(dict.fromkeys(base_cols + keys + ["val"]))]
    t = d[d["file_year"] >= 2015]
    if analysed_only:
        t = t[t["analysed"] == 1]
    t = t.reset_index(drop=True)

    t = t.merge(window_sums(d, keys, "wt"), on=keys + ["file_year"], how="left")
    t = t.merge(window_sums(d, keys + ["event_no"], "et"), on=keys + ["event_no", "file_year"], how="left")
    mu = ((t["wt_val"] - t["et_val"]) / (t["wt_svpt"] - t["et_svpt"])).to_numpy()

    ex_s, n_s = role_excess(t, d, "server_id", keys, "s", mu, k)
    ex_r, n_r = role_excess(t, d, "returner_id", keys, "r", mu, k)

    t["exp"] = np.clip(mu + ex_s + ex_r, 0.0, 1.0)
    t["resid"] = t["val"] - t["exp"] * t["svpt"]
    t["n_srv"], t["n_ret"] = n_s, n_r
    return t

In [ ]:
res_roof = expected_values_k(obs, "sv_won", ["surface", "indoor"])
print(len(res_roof), "rows | observed",
      round(float(100 * res_roof["val"].sum() / res_roof["svpt"].sum()), 2), "% | expected",
      round(float((100 * res_roof["exp"] * res_roof["svpt"]).sum() / res_roof["svpt"].sum()), 2), "%")
print("Service points won, relative to events with the same surface and roof")
print(speed_table(res_roof).to_string(index=False))

In [ ]:
cols = []
for k in [100, 250, 500, 1000, 2000]:
    r_ = expected_values_k(obs, "sv_won", ["surface", "indoor"], k=k)
    cols.append(speed_table(r_, B=100).set_index("name")["speed_pts"].rename(f"k={k}"))
print(pd.concat(cols, axis=1).round(2).to_string())

In [ ]:
def speed_by_year(res, B=500, min_matches=20, seed=1):
    m = res.groupby(["match_id", "event_no", "file_year"], as_index=False).agg(
        rows=("svpt", "size"), svpt=("svpt", "sum"), resid=("resid", "sum"))
    m = m[m["rows"] == 2]
    rng = np.random.default_rng(seed)
    out = []
    for (ev, yr), g in m.groupby(["event_no", "file_year"]):
        if len(g) < min_matches:
            continue
        s, r = g["svpt"].to_numpy(), g["resid"].to_numpy()
        idx = rng.integers(0, len(g), size=(B, len(g)))
        boots = 100 * r[idx].sum(axis=1) / s[idx].sum(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"event_no": ev, "year": yr, "matches": len(g),
                    "speed_pts": 100 * r.sum() / s.sum(), "half_width": (hi - lo) / 2})
    return pd.DataFrame(out).merge(tournament_names, on="event_no")

by_year = speed_by_year(res_roof)
print(by_year.pivot(index="name", columns="year", values="speed_pts").round(1).to_string())
print()
print(by_year.pivot(index="name", columns="year", values="half_width").round(1).to_string())

In [ ]:
by_year = speed_by_year(res_roof)
print(by_year.pivot(index="name", columns="year", values="speed_pts").round(1).to_string())
print()
print(by_year.pivot(index="name", columns="year", values="half_width").round(1).to_string())

In [ ]:
for name in ["pd", "np", "con", "obs", "expected_values_k", "speed_table", "tournament_names", "res_roof", "by_year"]:
    print(name, name in globals())

In [ ]:
meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Service points won above expectation (percentage points). Baseline: other events with the same "
               "surface and roof, same players, +/-1 year, shrinkage k=500. Interval: bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}

tbl_year = by_year.rename(columns={"year": "file_year", "speed_pts": "serve_excess_pp"})[
    ["event_no", "name", "file_year", "matches", "serve_excess_pp", "half_width"]].copy()
for k_, v_ in meta.items():
    tbl_year[k_] = v_
tbl_year.to_sql("serve_advantage_by_year", con, if_exists="replace", index=False)

pooled = speed_table(res_roof).rename(columns={"speed_pts": "serve_excess_pp"})
for k_, v_ in meta.items():
    pooled[k_] = v_
pooled.to_sql("serve_advantage_pooled", con, if_exists="replace", index=False)
con.commit()

print(pd.read_sql("SELECT COUNT(*) AS by_year_rows FROM serve_advantage_by_year", con).to_string(index=False))
print(pd.read_sql("SELECT COUNT(*) AS pooled_rows FROM serve_advantage_pooled", con).to_string(index=False))

In [ ]:
for name in ["pd", "np", "con"]:
    print(name, name in globals())
print(pd.read_sql("SELECT COUNT(*) AS n FROM serve_obs", con).iloc[0, 0], "rows in serve_obs")

In [ ]:
frames = []
for yr in range(2015, 2027):
    mm = 25 if yr == 2026 else 30
    c = season_card(yr, min_matches=mm, top=40).copy()
    c.insert(0, "season", yr)
    c["min_matches"] = mm
    c["as_of"] = pd.read_sql("SELECT MAX(tourney_start) AS d FROM matches WHERE file_year = :y",
                             con, params={"y": yr}).iloc[0, 0]
    frames.append(c)

cards = pd.concat(frames, ignore_index=True)
cards["source"] = "Calculated from TennisMyLife match data"
cards["licence"] = "Derived aggregate; non-commercial, see data/Sources.md"
cards["method"] = ("Per player and season (TML file year), ATP tour-level main-draw matches in our data. Elo v2 ratings; "
                   "wins above expected = sum of (result - expected probability); win % interval = Wilson 95%.")
cards["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
cards.to_sql("player_season_cards", con, if_exists="replace", index=False)
con.commit()

print(cards.groupby("season").size().to_string())

In [ ]:
def season_card(year, min_matches=30, top=15, sort_by="avg_elo"):
    p = {"y": year, "mm": min_matches}
    base = pd.read_sql("""
        SELECT e.player_id,
               COUNT(*) AS matches,
               SUM(e.result) AS wins,
               ROUND(AVG(e.elo_before)) AS avg_elo,
               ROUND(MAX(e.elo_after)) AS peak_elo,
               ROUND(AVG(o.elo_before)) AS avg_opp_elo,
               SUM(CASE WHEN e.expected < 0.5 THEN 1 ELSE 0 END) AS matches_as_underdog,
               SUM(CASE WHEN e.expected < 0.5 AND e.result = 1 THEN 1 ELSE 0 END) AS wins_as_underdog,
               ROUND(SUM(e.result - e.expected), 1) AS wins_above_expected
        FROM elo_long e
        JOIN elo_long o ON o.match_id = e.match_id AND o.player_id = e.opponent_id
        JOIN matches m ON m.match_id = e.match_id
        WHERE m.file_year = :y
        GROUP BY e.player_id
        HAVING COUNT(*) >= :mm
    """, con, params=p)

    titles = pd.read_sql("SELECT winner_id AS player_id, COUNT(*) AS titles FROM matches "
                         "WHERE round = 'F' AND file_year = :y GROUP BY winner_id", con, params={"y": year})
    lost = pd.read_sql("SELECT loser_id AS player_id, COUNT(*) AS finals_lost FROM matches "
                       "WHERE round = 'F' AND file_year = :y GROUP BY loser_id", con, params={"y": year})
    srv = pd.read_sql("SELECT server_id AS player_id, "
                      "ROUND(100.0 * SUM(sv_won) / SUM(svpt), 1) AS svc_pts_won_pct, "
                      "ROUND(100.0 * SUM(ace) / SUM(svpt), 1) AS aces_pct "
                      "FROM serve_obs WHERE file_year = :y GROUP BY server_id", con, params={"y": year})
    ret = pd.read_sql("SELECT returner_id AS player_id, "
                      "ROUND(100.0 * SUM(svpt - sv_won) / SUM(svpt), 1) AS ret_pts_won_pct "
                      "FROM serve_obs WHERE file_year = :y GROUP BY returner_id", con, params={"y": year})
    names = pd.read_sql("SELECT player_id, name FROM players", con)

    d = base
    for t in (titles, lost, srv, ret, names):
        d = d.merge(t, on="player_id", how="left")
    d[["titles", "finals_lost"]] = d[["titles", "finals_lost"]].fillna(0).astype(int)

    z = 1.96
    n, pw = d["matches"], d["wins"] / d["matches"]
    center = (pw + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(pw * (1 - pw) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    d["win_pct"] = (100 * pw).round(1)
    d["win_pct_low"] = (100 * (center - half)).round(1)
    d["win_pct_high"] = (100 * (center + half)).round(1)

    cols = ["name", "matches", "wins", "win_pct", "win_pct_low", "win_pct_high", "titles", "finals_lost",
            "avg_elo", "peak_elo", "avg_opp_elo", "wins_as_underdog", "matches_as_underdog",
            "wins_above_expected", "svc_pts_won_pct", "ret_pts_won_pct", "aces_pct"]
    return d.sort_values(sort_by, ascending=False).head(top)[cols]

In [ ]:
print(season_card(2024, top=3).to_string(index=False))

In [ ]:
frames = []
for yr in range(2015, 2027):
    mm = 25 if yr == 2026 else 30
    c = season_card(yr, min_matches=mm, top=40).copy()
    c.insert(0, "season", yr)
    c["min_matches"] = mm
    c["as_of"] = pd.read_sql("SELECT MAX(tourney_start) AS d FROM matches WHERE file_year = :y",
                             con, params={"y": yr}).iloc[0, 0]
    frames.append(c)

cards = pd.concat(frames, ignore_index=True)
cards["source"] = "Calculated from TennisMyLife match data"
cards["licence"] = "Derived aggregate; non-commercial, see data/Sources.md"
cards["method"] = ("Per player and season (TML file year), ATP tour-level main-draw matches in our data. Elo v2 ratings; "
                   "wins above expected = sum of (result - expected probability); win % interval = Wilson 95%.")
cards["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
cards.to_sql("player_season_cards", con, if_exists="replace", index=False)
con.commit()

print(cards.groupby("season").size().to_string())

In [ ]:
import numpy as np

con.execute("DROP VIEW IF EXISTS server_rows")
con.execute("""
CREATE VIEW server_rows AS
SELECT match_id, file_year, event_no, surface, indoor, winner_id AS server_id,
       w_svpt AS svpt, w_ace AS ace, w_df AS df, w_1stIn AS first_in, w_1stWon AS first_won,
       w_2ndWon AS second_won, w_SvGms AS sv_games, w_bpFaced AS bp_faced, w_bpSaved AS bp_saved
FROM matches
WHERE retired = 0 AND w_svpt > 0 AND w_ace IS NOT NULL AND w_df IS NOT NULL AND w_1stIn IS NOT NULL
  AND w_1stWon IS NOT NULL AND w_2ndWon IS NOT NULL AND w_SvGms IS NOT NULL
  AND w_bpFaced IS NOT NULL AND w_bpSaved IS NOT NULL
UNION ALL
SELECT match_id, file_year, event_no, surface, indoor, loser_id,
       l_svpt, l_ace, l_df, l_1stIn, l_1stWon, l_2ndWon, l_SvGms, l_bpFaced, l_bpSaved
FROM matches
WHERE retired = 0 AND l_svpt > 0 AND l_ace IS NOT NULL AND l_df IS NOT NULL AND l_1stIn IS NOT NULL
  AND l_1stWon IS NOT NULL AND l_2ndWon IS NOT NULL AND l_SvGms IS NOT NULL
  AND l_bpFaced IS NOT NULL AND l_bpSaved IS NOT NULL
""")
con.commit()

sr = pd.read_sql("SELECT * FROM server_rows WHERE file_year >= 2015", con)
sr["spw_num"] = sr["first_won"] + sr["second_won"]
sr["second_in_den"] = sr["svpt"] - sr["first_in"]
sr["hold_num"] = sr["sv_games"] - (sr["bp_faced"] - sr["bp_saved"])
ok = sr["hold_num"] >= 0
sr["hold_ok"] = np.where(ok, sr["hold_num"], 0)
sr["games_ok"] = np.where(ok, sr["sv_games"], 0)
print(len(sr), "server rows | negative holds:", int((~ok).sum()),
      "| second-serve denominators <= 0:", int((sr["second_in_den"] <= 0).sum()))
print("columns present:", "hold_ok" in sr.columns, "games_ok" in sr.columns)

In [ ]:
info = pd.read_sql("SELECT match_id, best_of, minutes, score FROM matches WHERE file_year >= 2015", con)
num_cols = ["spw_num", "ace", "df", "first_in", "first_won", "second_won", "hold_num", "svpt", "sv_games","second_in_den", "hold_ok", "games_ok"]
pm = sr.groupby(["match_id", "file_year", "event_no", "surface", "indoor"], as_index=False)[num_cols].sum()
pm = pm.merge(info, on="match_id", how="left")
pm["is_paris"] = pm["event_no"].astype(str) == "352"
bo3 = pm["best_of"] == 3
pm["min_den"] = (bo3 & pm["minutes"].notna()).astype(int)
pm["min_num"] = np.where(pm["min_den"] == 1, pm["minutes"], 0.0)
pm["tb_den"] = bo3.astype(int)
pm["tb_num"] = (bo3 & pm["score"].str.contains(r"7-6|6-7", na=False)).astype(int)

metrics = {
    "service_pts_won_pct": ("spw_num", "svpt", 100),
    "aces_pct": ("ace", "svpt", 100),
    "double_faults_pct": ("df", "svpt", 100),
    "first_serve_in_pct": ("first_in", "svpt", 100),
    "first_serve_pts_won_pct": ("first_won", "first_in", 100),
    "second_serve_pts_won_pct": ("second_won", "second_in_den", 100),
    "service_games_held_pct": ("hold_ok", "games_ok", 100),
    "minutes_per_match_bo3": ("min_num", "min_den", 1),
    "matches_with_tiebreak_pct_bo3": ("tb_num", "tb_den", 100),
}


def boot_ratio(num, den, scale, rng, B=300):
    idx = rng.integers(0, len(num), size=(B, len(num)))
    return np.percentile(scale * num[idx].sum(axis=1) / den[idx].sum(axis=1), [2.5, 97.5])


latest = pd.read_sql("SELECT MAX(tourney_start) AS d FROM matches", con).iloc[0, 0]
ps = pd.read_sql("SELECT file_year, MIN(tourney_start) AS s FROM matches WHERE event_no = '352' GROUP BY file_year", con)
partial_years = set(ps[pd.to_datetime(ps["s"]) >= pd.Timestamp(latest) - pd.Timedelta(days=6)]["file_year"])

rng = np.random.default_rng(1)
rows = []
for yr in [y for y in sorted(pm["file_year"].unique()) if y <= 2025]:
    py = pm[pm["file_year"] == yr]
    groups = {
        "Paris Masters": py["is_paris"],
        "Rest of the tour": ~py["is_paris"],
        "Other indoor hard events": (~py["is_paris"]) & (py["indoor"] == "I") & (py["surface"] == "Hard"),
    }
    for gname, mask in groups.items():
        g = py[mask]
        if len(g) < (10 if gname == "Paris Masters" else 30):
            continue
        for metric, (nc, dc, scale) in metrics.items():
            num, den = g[nc].to_numpy(float), g[dc].to_numpy(float)
            if den.sum() == 0:
                continue
            lo, hi = boot_ratio(num, den, scale, rng)
            rows.append({"metric": metric, "group": gname, "file_year": int(yr),
                         "value": scale * num.sum() / den.sum(), "ci_low": float(lo), "ci_high": float(hi),
                         "matches": len(g), "partial": int(gname == "Paris Masters" and yr in partial_years)})

by_year_tbl = pd.DataFrame(rows)
print("latest tournament start in the data:", latest, "| partial Paris years:", sorted(partial_years))
print(by_year_tbl[by_year_tbl["metric"] == "service_pts_won_pct"]
      .pivot(index="file_year", columns="group", values="value").round(2).to_string())

In [ ]:
def pooled_difference(pm, years, B=300, seed=2):
    rng = np.random.default_rng(seed)
    sel = pm[pm["file_year"].isin(years)]
    p, r = sel[sel["is_paris"]], sel[~sel["is_paris"]]
    P = {m: (p[nc].to_numpy(float), p[dc].to_numpy(float)) for m, (nc, dc, sc) in metrics.items()}
    R = {m: (r[nc].to_numpy(float), r[dc].to_numpy(float)) for m, (nc, dc, sc) in metrics.items()}
    diffs = {m: [] for m in metrics}
    for _ in range(B):
        ip, ir = rng.integers(0, len(p), len(p)), rng.integers(0, len(r), len(r))
        for m, (nc, dc, sc) in metrics.items():
            diffs[m].append(sc * P[m][0][ip].sum() / P[m][1][ip].sum() - sc * R[m][0][ir].sum() / R[m][1][ir].sum())
    out = []
    for m, (nc, dc, sc) in metrics.items():
        pv, rv = sc * P[m][0].sum() / P[m][1].sum(), sc * R[m][0].sum() / R[m][1].sum()
        lo, hi = np.percentile(diffs[m], [2.5, 97.5])
        out.append({"metric": m, "paris": pv, "rest_of_tour": rv, "difference": pv - rv, "ci_low": float(lo),
                    "ci_high": float(hi), "unit": "minutes" if sc == 1 else "percentage points",
                    "paris_matches": len(p), "rest_matches": len(r),
                    "years": f"{min(years)}-{max(years)}"})
    return pd.DataFrame(out)


pooled_tbl = pooled_difference(pm, list(range(2015, 2026)))
print(pooled_tbl[["metric", "paris", "rest_of_tour", "difference", "ci_low", "ci_high"]].round(2).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Per statistic: sums over servers' rows (retired matches excluded). Bands: 95% bootstrap over matches. "
               "Rest of the tour = all other tour-level events that year. Minutes and tiebreaks: best-of-3 matches only. "
               "Service games held derived from break points faced and saved."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    by_year_tbl[k_] = v_
    pooled_tbl[k_] = v_
by_year_tbl.to_sql("paris_characteristics_by_year", con, if_exists="replace", index=False)
pooled_tbl.to_sql("paris_vs_tour_pooled", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(by_year_tbl), "yearly rows,", len(pooled_tbl), "pooled rows")

In [ ]:
bad = sr[sr["hold_num"] < 0]
print(bad.groupby("file_year").size().to_string())
print("in Paris:", int((bad["event_no"].astype(str) == "352").sum()))

In [ ]:
for yrs, label in [(list(range(2015, 2026)), "2015-2025"),
                   ([y for y in range(2015, 2026) if y not in (2017, 2018)], "without 2017-2018")]:
    t = pooled_difference(pm, yrs)
    print(label)
    print(t[t["metric"] == "service_games_held_pct"][["paris", "rest_of_tour", "difference", "ci_low", "ci_high"]].round(2).to_string(index=False))

In [ ]:
print(pd.read_sql("""
    SELECT metric, ROUND(paris, 2) AS paris, ROUND(rest_of_tour, 2) AS rest_of_tour, ROUND(difference, 2) AS difference
    FROM paris_vs_tour_pooled WHERE metric = 'service_games_held_pct'
""", con).to_string(index=False))

In [ ]:
tm = pd.read_sql("SELECT event_no, name FROM tournaments WHERE level = 'M'", con)
tm["event_no"] = tm["event_no"].astype(str)
pmm = pm.assign(event_no=pm["event_no"].astype(str)).merge(tm, on="event_no")
pmm = pmm[pmm["file_year"] <= 2025]

rng = np.random.default_rng(3)
rows = []


def add_row(g, label, yr):
    num, den = g["hold_ok"].to_numpy(float), g["games_ok"].to_numpy(float)
    if len(g) < 20 or den.sum() == 0:
        return
    lo, hi = boot_ratio(num, den, 100, rng)
    rows.append({"name": label, "file_year": int(yr), "matches": len(g),
                 "hold_pct": 100 * num.sum() / den.sum(), "ci_low": float(lo), "ci_high": float(hi)})


for (nm, yr), g in pmm.groupby(["name", "file_year"]):
    add_row(g, nm, yr)
for yr, g in pmm.groupby("file_year"):
    add_row(g, "All Masters 1000 (pooled)", yr)

hold_tbl = pd.DataFrame(rows)
print(hold_tbl.pivot(index="file_year", columns="name", values="hold_pct").round(1).to_string())

meta2 = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Service games held = (service games - (break points faced - break points saved)) / service games, "
               "summed over matches; rows with impossible counts excluded; retired matches excluded. "
               "Bars: 95% bootstrap over matches, at least 20 matches per tournament-year."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta2.items():
    hold_tbl[k_] = v_
hold_tbl.to_sql("masters_service_games_held", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(hold_tbl), "rows")

In [ ]:
def boot_ratio(num, den, scale, rng, B=300):
    idx = rng.integers(0, len(num), size=(B, len(num)))
    return np.percentile(scale * num[idx].sum(axis=1) / den[idx].sum(axis=1), [2.5, 97.5])


valid = (sr["hold_num"] >= 0) & (sr["bp_saved"] <= sr["bp_faced"])
sr["bp_faced_ok"] = np.where(valid, sr["bp_faced"], 0)
sr["bp_conv_ok"] = np.where(valid, sr["bp_faced"] - sr["bp_saved"], 0)
sr["games_bp_ok"] = np.where(valid, sr["sv_games"], 0)
pm_bp = sr.groupby(["match_id", "file_year", "event_no", "surface", "indoor"], as_index=False)[
    ["bp_faced_ok", "bp_conv_ok", "games_bp_ok"]].sum()
pm_bp["is_paris"] = pm_bp["event_no"].astype(str) == "352"
pm_bp = pm_bp[pm_bp["file_year"] <= 2025]

fav = pd.read_sql("""
    SELECT m.match_id, m.file_year, m.event_no, m.surface, m.indoor, e.expected AS p_winner
    FROM elo_long e JOIN matches m ON m.match_id = e.match_id
    WHERE e.result = 1 AND m.file_year BETWEEN 2015 AND 2025
""", con)
fav = fav[fav["p_winner"] != 0.5].copy()
fav["p_fav"] = np.maximum(fav["p_winner"], 1 - fav["p_winner"])
fav["fav_won"] = (fav["p_winner"] > 0.5).astype(float)
fav["fav_gap"] = fav["fav_won"] - fav["p_fav"]
fav["is_paris"] = fav["event_no"].astype(str) == "352"
fav["one"] = 1.0
for nm, lo_, hi_ in [("b1", 0.5, 0.6), ("b2", 0.6, 0.75), ("b3", 0.75, 1.01)]:
    inb = (fav["p_fav"] > lo_) & (fav["p_fav"] <= hi_)
    fav["n_" + nm] = inb.astype(float)
    fav["w_" + nm] = (inb & (fav["fav_won"] == 1)).astype(float)

bp_mets = {
    "break_points_converted_pct": ("bp_conv_ok", "bp_faced_ok", 100),
    "break_points_per_100_service_games": ("bp_faced_ok", "games_bp_ok", 100),
}
fav_mets = {
    "favourite_win_pct": ("fav_won", "one", 100),
    "avg_favourite_probability_pct": ("p_fav", "one", 100),
    "favourite_wins_above_expected_pct": ("fav_gap", "one", 100),
    "favourite_win_pct_50_60": ("w_b1", "n_b1", 100),
    "favourite_win_pct_60_75": ("w_b2", "n_b2", 100),
    "favourite_win_pct_75_plus": ("w_b3", "n_b3", 100),
    "matches_with_75_plus_favourite_pct": ("n_b3", "one", 100),
}
print("rows with more break points saved than faced:", int((sr["bp_saved"] > sr["bp_faced"]).sum()))
print("matches with break-point data:", len(pm_bp), "| Paris:", int(pm_bp["is_paris"].sum()))
print("matches in the favourite table:", len(fav), "| Paris:", int(fav["is_paris"].sum()))

In [ ]:
def yearly_series(df, mets, years, seed=7, B=300, min_paris=10, min_other=30):
    rng = np.random.default_rng(seed)
    rows = []
    for yr in years:
        dy = df[df["file_year"] == yr]
        groups = {"Paris Masters": dy["is_paris"],
                  "Rest of the tour": ~dy["is_paris"],
                  "Other indoor hard events": (~dy["is_paris"]) & (dy["indoor"] == "I") & (dy["surface"] == "Hard")}
        for gname, mask in groups.items():
            g = dy[mask]
            if len(g) < (min_paris if gname == "Paris Masters" else min_other):
                continue
            for metric, (nc, dc, sc) in mets.items():
                num, den = g[nc].to_numpy(float), g[dc].to_numpy(float)
                if den.sum() == 0:
                    continue
                lo, hi = boot_ratio(num, den, sc, rng, B)
                rows.append({"metric": metric, "group": gname, "file_year": int(yr), "value": sc * num.sum() / den.sum(),
                             "ci_low": float(lo), "ci_high": float(hi), "matches": len(g)})
    return pd.DataFrame(rows)


def pooled_vs(df, mets, years, ref, B=300, seed=5):
    rng = np.random.default_rng(seed)
    sel = df[df["file_year"].isin(years)]
    p = sel[sel["is_paris"]]
    rest = sel[~sel["is_paris"]]
    r = rest if ref == "Rest of the tour" else rest[(rest["indoor"] == "I") & (rest["surface"] == "Hard")]
    P = {m: (p[a].to_numpy(float), p[b].to_numpy(float)) for m, (a, b, s) in mets.items()}
    R = {m: (r[a].to_numpy(float), r[b].to_numpy(float)) for m, (a, b, s) in mets.items()}
    diffs = {m: [] for m in mets}
    for _ in range(B):
        ip, ir = rng.integers(0, len(p), len(p)), rng.integers(0, len(r), len(r))
        for m, (a, b, s) in mets.items():
            diffs[m].append(s * P[m][0][ip].sum() / P[m][1][ip].sum() - s * R[m][0][ir].sum() / R[m][1][ir].sum())
    out = []
    for m, (a, b, s) in mets.items():
        pv, rv = s * P[m][0].sum() / P[m][1].sum(), s * R[m][0].sum() / R[m][1].sum()
        lo, hi = np.percentile(diffs[m], [2.5, 97.5])
        out.append({"metric": m, "reference": ref, "paris": pv, "reference_value": rv, "difference": pv - rv,
                    "ci_low": float(lo), "ci_high": float(hi),
                    "unit": "break points per 100 service games" if m == "break_points_per_100_service_games" else "percentage points",
                    "paris_matches": len(p), "reference_matches": len(r), "years": f"{min(years)}-{max(years)}"})
    return pd.DataFrame(out)


years = list(range(2015, 2026))
yearly = pd.concat([yearly_series(pm_bp, bp_mets, years), yearly_series(fav, fav_mets, years)], ignore_index=True)
pooled = pd.concat([pooled_vs(pm_bp, bp_mets, years, "Rest of the tour"),
                    pooled_vs(fav, fav_mets, years, "Rest of the tour"),
                    pooled_vs(pm_bp, bp_mets, years, "Other indoor hard events"),
                    pooled_vs(fav, fav_mets, years, "Other indoor hard events")], ignore_index=True)

cols = ["metric", "reference", "paris", "reference_value", "difference", "ci_low", "ci_high"]
print(pooled[cols].round(2).to_string(index=False))

In [ ]:
meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Break points: sums over servers' rows, retired matches and rows with impossible counts excluded. "
               "Favourite = the player rated higher by our Elo v2 (calibrated probability before the match); retirements count. "
               "Bands: 95% bootstrap over matches. Completed editions 2015-2025."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    yearly[k_] = v_
    pooled[k_] = v_
yearly.to_sql("paris_extra_by_year", con, if_exists="replace", index=False)
pooled.to_sql("paris_extra_pooled", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(yearly), "yearly rows,", len(pooled), "pooled rows")

In [ ]:
t14 = pd.read_sql("SELECT event_no, name, indoor_outdoor FROM tournaments WHERE surface = 'Hard'", con)
t14["event_no"] = t14["event_no"].astype(str)
pmh = pm.assign(event_no=pm["event_no"].astype(str)).merge(t14, on="event_no")
pmh = pmh[pmh["file_year"] <= 2025]
print(pmh.groupby("name").size().sort_values(ascending=False).to_string())

B = 500
rng = np.random.default_rng(11)


def boot_dist(n, d):
    idx = rng.integers(0, len(n), size=(B, len(n)))
    return 100 * n[idx].sum(axis=1) / d[idx].sum(axis=1)


arr = {nm: (g["ace"].to_numpy(float), g["svpt"].to_numpy(float)) for nm, g in pmh.groupby("name")}
others = pmh[pmh["event_no"] != "352"]
OTH = "Other hard-court events (pooled)"
arr[OTH] = (others["ace"].to_numpy(float), others["svpt"].to_numpy(float))
dist = {nm: boot_dist(n, d) for nm, (n, d) in arr.items()}
roof = dict(zip(t14["name"], t14["indoor_outdoor"]))
cnt = pmh.groupby("name").size().to_dict()
cnt[OTH] = len(others)
paris_val = 100 * arr["Paris Masters"][0].sum() / arr["Paris Masters"][1].sum()

rows = []
for nm, (n, d) in arr.items():
    val = 100 * n.sum() / d.sum()
    lo, hi = np.percentile(dist[nm] - dist["Paris Masters"], [2.5, 97.5])
    rows.append({"name": nm, "matches": cnt[nm], "aces_pct": val, "diff_vs_paris": val - paris_val,
                 "ci_low": float(lo), "ci_high": float(hi), "indoor_outdoor": roof.get(nm, "")})
pooled_hard = pd.DataFrame(rows).sort_values("aces_pct", ascending=False)
print(pooled_hard.round(2).to_string(index=False))

yr_rows = []
for yr in range(2015, 2026):
    py = pmh[pmh["file_year"] == yr]
    for label, mask in [("Paris Masters", py["event_no"] == "352"),
                        ("Other hard-court events", py["event_no"] != "352")]:
        g = py[mask]
        if len(g) < 10:
            continue
        yr_rows.append({"group": label, "file_year": yr,
                        "aces_pct": 100 * g["ace"].sum() / g["svpt"].sum(), "matches": len(g)})
by_year_hard = pd.DataFrame(yr_rows)

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Aces as a share of service points, summed over servers' rows (retired matches and rows with missing "
               "statistics excluded). Hard-court events among the 14 analysed events, completed editions 2015-2025. "
               "diff_vs_paris: event minus Paris, 95% bootstrap interval over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    pooled_hard[k_] = v_
    by_year_hard[k_] = v_
pooled_hard.to_sql("aces_hard_events_pooled", con, if_exists="replace", index=False)
by_year_hard.to_sql("aces_paris_vs_hard_by_year", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(pooled_hard), "pooled rows,", len(by_year_hard), "yearly rows")

In [ ]:
print(pd.read_sql("""SELECT (SELECT COUNT(*) FROM matches) AS matches,
                            (SELECT COUNT(*) FROM elo_long)/2 AS elo_matches""", con).to_string(index=False))

In [ ]:
print(pd.read_sql("""
    SELECT tourney_start, tourney_name, tourney_level, COUNT(*) AS matches
    FROM matches WHERE file_year = 2026 AND tourney_start >= '2026-10-12'
    GROUP BY tourney_start, tourney_name, tourney_level ORDER BY tourney_start
""", con).to_string(index=False))

In [ ]:
PLAYER = "Novak Djokovic"
FIRST, LAST = 2007, 2025

ids = pd.read_sql("SELECT player_id, name FROM players WHERE name LIKE :n", con,
                  params={"n": f"%{PLAYER.split()[-1]}%"})
print(ids.to_string(index=False))
pid = ids.loc[ids["name"] == PLAYER, "player_id"].iloc[0]

hard = pd.read_sql("SELECT event_no, name, indoor_outdoor FROM tournaments WHERE surface = 'Hard'", con)
hard["event_no"] = hard["event_no"].astype(str)

d = pd.read_sql("""
    SELECT e.match_id, m.file_year, m.event_no, m.round, m.round_order, e.result, e.expected
    FROM elo_long e JOIN matches m ON m.match_id = e.match_id
    WHERE e.player_id = :p AND m.file_year BETWEEN :a AND :b
""", con, params={"p": pid, "a": FIRST, "b": LAST})
d["event_no"] = d["event_no"].astype(str)
d = d.merge(hard, on="event_no")
d["wae"] = d["result"] - d["expected"]
d["is_paris"] = d["event_no"] == "352"
print(len(d), "matches at hard-court events |", int(d["is_paris"].sum()), "at Paris")

rng = np.random.default_rng(21)
B = 2000


def boot_mean(x):
    idx = rng.integers(0, len(x), size=(B, len(x)))
    return x[idx].mean(axis=1)


rows, dist = [], {}
for nm, g in d.groupby("name"):
    x = g["wae"].to_numpy(float)
    dist[nm] = boot_mean(x)
    lo, hi = np.percentile(dist[nm], [2.5, 97.5])
    rows.append({"player": PLAYER, "name": nm, "matches": len(g), "wins": int(g["result"].sum()),
                 "win_pct": 100 * g["result"].mean(), "wae_per_100": 100 * x.mean(),
                 "ci_low": 100 * lo, "ci_high": 100 * hi, "indoor_outdoor": g["indoor_outdoor"].iloc[0],
                 "diff_paris_minus_others": np.nan, "diff_ci_low": np.nan, "diff_ci_high": np.nan})

oth = d[~d["is_paris"]]
xo = oth["wae"].to_numpy(float)
do = boot_mean(xo)
lo, hi = np.percentile(do, [2.5, 97.5])
dlo, dhi = np.percentile(dist["Paris Masters"] - do, [2.5, 97.5])
paris_mean = d[d["is_paris"]]["wae"].mean()
rows.append({"player": PLAYER, "name": "Other hard-court events (pooled)", "matches": len(oth),
             "wins": int(oth["result"].sum()), "win_pct": 100 * oth["result"].mean(),
             "wae_per_100": 100 * xo.mean(), "ci_low": 100 * lo, "ci_high": 100 * hi, "indoor_outdoor": "",
             "diff_paris_minus_others": 100 * (paris_mean - xo.mean()),
             "diff_ci_low": 100 * dlo, "diff_ci_high": 100 * dhi})
events_tbl = pd.DataFrame(rows).sort_values("wae_per_100", ascending=False)
print(events_tbl.round(1).drop(columns=["player"]).to_string(index=False))

In [ ]:
STAGE = {"R128": 0, "R64": 1, "R32": 2, "R16": 3, "QF": 4, "SF": 5, "F": 6}
par = d[d["is_paris"]]
yr_rows = []
for yr, g in par.groupby("file_year"):
    last = g.sort_values("round_order").iloc[-1]
    if last["round"] == "F" and last["result"] == 1:
        res, stage = "Champion", 7
    elif last["result"] == 1:
        res, stage = f"Won {last['round']}, did not play on", STAGE[last["round"]] + 1
    else:
        res = {"F": "Runner-up"}.get(last["round"], "Lost in " + last["round"])
        stage = STAGE[last["round"]]
    yr_rows.append({"player": PLAYER, "file_year": int(yr), "matches": len(g), "wins": int(g["result"].sum()),
                    "result": res, "stage": stage})
years_tbl = pd.DataFrame(yr_rows)
print(years_tbl.drop(columns=["player"]).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Wins above expected = sum of (result - expected probability) from our Elo v2, per 100 matches; "
               "hard-court events among the 14 analysed events, 2007-2025; retirements count; bars: 95% bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    events_tbl[k_] = v_
    years_tbl[k_] = v_
events_tbl.to_sql("player_hard_events", con, if_exists="replace", index=False)
years_tbl.to_sql("player_paris_by_year", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(events_tbl), "event rows,", len(years_tbl), "yearly rows")

In [ ]:
FIRST, LAST = 2007, 2025
py_tbl = pd.read_sql("SELECT file_year, result FROM player_paris_by_year", con)
paris_years = sorted(py_tbl["file_year"].tolist())
champ = set(py_tbl.loc[py_tbl["result"] == "Champion", "file_year"])

mm = pd.read_sql("""SELECT match_id, file_year, event_no, winner_id FROM matches
                    WHERE (winner_id = :p OR loser_id = :p) AND file_year BETWEEN :a AND :b""",
                 con, params={"p": pid, "a": FIRST, "b": LAST})
mm = mm[mm["file_year"].isin(paris_years)].copy()
mm["won"] = (mm["winner_id"] == pid).astype(float)
mm["is_paris"] = mm["event_no"].astype(str) == "352"
mm["one"] = 1.0
ids = set(mm["match_id"])

sv = pd.read_sql("SELECT * FROM server_rows WHERE file_year BETWEEN :a AND :b", con, params={"a": FIRST, "b": LAST})
sv = sv[sv["match_id"].isin(ids)]
mine = sv[sv["server_id"] == pid].copy()
opp = sv[sv["server_id"] != pid].copy()

mine["sv_n"] = mine["first_won"] + mine["second_won"]
mine["sv_d"] = mine["svpt"]
mine["ace_n"] = mine["ace"]
hold_num = mine["sv_games"] - (mine["bp_faced"] - mine["bp_saved"])
okh = hold_num >= 0
mine["hold_n"] = np.where(okh, hold_num, 0)
mine["hold_d"] = np.where(okh, mine["sv_games"], 0)

opp["ret_n"] = opp["svpt"] - (opp["first_won"] + opp["second_won"])
opp["ret_d"] = opp["svpt"]
okb = opp["bp_saved"] <= opp["bp_faced"]
opp["bpc_n"] = np.where(okb, opp["bp_faced"] - opp["bp_saved"], 0)
opp["bpc_d"] = np.where(okb, opp["bp_faced"], 0)

pmx = (mm.merge(mine[["match_id", "sv_n", "sv_d", "ace_n", "hold_n", "hold_d"]], on="match_id", how="left")
         .merge(opp[["match_id", "ret_n", "ret_d", "bpc_n", "bpc_d"]], on="match_id", how="left").fillna(0))

M = {
    "win_pct": ("won", "one", 100),
    "service_pts_won_pct": ("sv_n", "sv_d", 100),
    "return_pts_won_pct": ("ret_n", "ret_d", 100),
    "aces_pct": ("ace_n", "sv_d", 100),
    "service_games_held_pct": ("hold_n", "hold_d", 100),
    "break_points_converted_pct": ("bpc_n", "bpc_d", 100),
}

rows = []
for yr in paris_years:
    for label, mask in [("Paris Masters", pmx["is_paris"]), ("Rest of his season", ~pmx["is_paris"])]:
        g = pmx[(pmx["file_year"] == yr) & mask]
        for metric, (nc, dc, sc) in M.items():
            den = g[dc].sum()
            if den > 0:
                rows.append({"player": PLAYER, "metric": metric, "group": label, "file_year": int(yr),
                             "value": sc * g[nc].sum() / den, "matches": len(g), "champion": int(yr in champ)})
season_tbl = pd.DataFrame(rows)

rng = np.random.default_rng(31)


def boot_ratio_one(g, nc, dc, sc, B=500):
    n, d = g[nc].to_numpy(float), g[dc].to_numpy(float)
    idx = rng.integers(0, len(n), size=(B, len(n)))
    return sc * n[idx].sum(axis=1) / d[idx].sum(axis=1)


pp, rr = pmx[pmx["is_paris"]], pmx[~pmx["is_paris"]]
pooled_rows = []
for metric, (nc, dc, sc) in M.items():
    pv, rv = sc * pp[nc].sum() / pp[dc].sum(), sc * rr[nc].sum() / rr[dc].sum()
    lo, hi = np.percentile(boot_ratio_one(pp, nc, dc, sc) - boot_ratio_one(rr, nc, dc, sc), [2.5, 97.5])
    pooled_rows.append({"player": PLAYER, "metric": metric, "paris": pv, "rest_of_season": rv, "difference": pv - rv,
                        "ci_low": float(lo), "ci_high": float(hi),
                        "paris_matches": len(pp), "rest_matches": len(rr), "years": f"{min(paris_years)}-{max(paris_years)}"})
season_pooled = pd.DataFrame(pooled_rows)
print(len(pp), "Paris matches in", len(paris_years), "years |", len(rr), "other matches those years")
print(season_pooled[["metric", "paris", "rest_of_season", "difference", "ci_low", "ci_high"]].round(1).to_string(index=False))

meta = {
    "source": "Calculated from TennisMyLife match data",
    "licence": "Derived aggregate; non-commercial, see data/Sources.md",
    "method": ("Per year he played the Paris Masters: Paris vs the rest of his tour-level season in our data (Olympics, Davis Cup "
               "and team events not included). Retired matches excluded from serve and return statistics; service games held "
               "and break points exclude rows with impossible counts. Pooled difference: 95% bootstrap over matches."),
    "computed_on": pd.Timestamp.today().strftime("%Y-%m-%d"),
}
for k_, v_ in meta.items():
    season_tbl[k_] = v_
    season_pooled[k_] = v_
season_tbl.to_sql("player_season_vs_paris", con, if_exists="replace", index=False)
season_pooled.to_sql("player_season_vs_paris_pooled", con, if_exists="replace", index=False)
con.commit()
print("saved:", len(season_tbl), "yearly rows,", len(season_pooled), "pooled rows")